# Weekly Social Media Reporting Automation

Consolidate downloaded Facebook, Instagram, TikTok, X, and YouTube exports into one CSV using pandas. Extract show labels from post titles and map their sources using a reference table. Human validation remains part of the workflow.

## Important: internal reporting context

> **This workflow reflects reporting requirements and conventions agreed through internal communication with the relevant teams. It is a portfolio example of an internally aligned process, not a universal reference for social media measurement.**

Most metric mappings and calculations follow the definitions in the relevant platform exports. Additional choices, including interaction formulas, labeling rules, zero defaults, and the shared output structure, reflect the internal reporting conventions used for this workflow.

Column names are standardized to simplify consolidation, review, and internal approval of the weekly report. **A shared column name does not mean that the underlying metrics have identical definitions or can be compared directly across platforms.** For example, Facebook three-second video views and YouTube views follow different measurement definitions.

Other organizations may use different reporting objectives, approved formulas, source exports, and treatment of unavailable values. Anyone adapting this code should confirm current platform definitions and agree the mapping and calculation rules with their own teams before using the output for reporting or decisions.

## Setup

Run this notebook in an environment with pandas and IPython installed. Start from a folder containing `All sources.csv` and a `Weekly Report` directory with `Facebook Data.csv`, `Instagram Data.csv`, `TikTok Data.csv`, `X Data.csv`, and `YouTube Data.csv`.

The reference table requires `Series Title` and `Sources` columns. Platform exports must contain the text columns referenced in their transformation sections. Optional numerical columns default to zero.

## Reporting conventions

- Missing or invalid numerical values become zero for compatibility with the original report. Unavailable metrics and measured zeros are not distinguished in the output.
- Show and source labels are rule-based. Blank labels remain blank for manual review.
- Platform metric definitions differ even when they share output column names.
- This workflow processes exports; it does not download data from platform APIs.
- The output format, calculations, column order, filenames, and label logic match the original notebook.

## Publishing

Saved outputs have been cleared. Share only permitted inputs and lookup tables, or provide clearly labeled synthetic examples. This notebook has been checked for unchanged executable logic, but has not been run against the original input CSVs in this review.


## Imports


In [ ]:
import re
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import FileLink, display




## 1. Files and folders


In [ ]:
# ==================================================
# 1. Files and folders
# ==================================================

BASE_FOLDER = Path(".")
INPUT_FOLDER = BASE_FOLDER / "Weekly Report"
OUTPUT_ROOT = BASE_FOLDER / "Weekly Report"

FILES = {
    "Facebook": INPUT_FOLDER / "Facebook Data.csv",
    "Instagram": INPUT_FOLDER / "Instagram Data.csv",
    "TikTok": INPUT_FOLDER / "TikTok Data.csv",
    "X": INPUT_FOLDER / "X Data.csv",
    "YouTube": INPUT_FOLDER / "YouTube Data.csv",
}

SOURCES_FILE = BASE_FOLDER / "All sources.csv"

OUTPUT_COLUMNS = [
    "Date",
    "Source",
    "Show",
    "platform",
    "Post type",
    "Title",
    "URL",
    "IMP",
    "Reach",
    "Video view count",
    "Total interactions",
    "Like",
    "comments",
    "share",
    "Save",
    "Video length (sec)",
    "3-second video views from Boosted posts",
    "3-second video views from Organic posts",
]

NUMERIC_COLUMNS = [
    "IMP",
    "Reach",
    "Video view count",
    "Total interactions",
    "Like",
    "comments",
    "share",
    "Save",
    "Video length (sec)",
    "3-second video views from Boosted posts",
    "3-second video views from Organic posts",
]




## 2. General helper functions


In [ ]:
# ==================================================
# 2. General helper functions
# ==================================================

required_files = [SOURCES_FILE, *FILES.values()]
missing_files = [str(path) for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "The following required files were not found:\n"
        + "\n".join(missing_files)
    )


def read_csv_file(path):
    """Read all columns as text to preserve IDs and Arabic text."""
    return pd.read_csv(
        path,
        dtype=str,
        encoding="utf-8-sig",
    ).fillna("")


# Reporting convention: unavailable columns, blank values, and unparseable
# numbers become zero. Zero therefore does not always mean a measured zero.
# Preserve this convention for compatibility with the existing weekly report.
def number_column(dataframe, column_name, default=0):
    """Return a numeric Series; missing columns and invalid values become 0."""
    if column_name not in dataframe.columns:
        return pd.Series(default, index=dataframe.index, dtype="float64")

    cleaned = (
        dataframe[column_name]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
    )

    return pd.to_numeric(cleaned, errors="coerce").fillna(default)


MONTH_NUMBERS = {
    "jan": 1, "january": 1,
    "feb": 2, "february": 2,
    "mar": 3, "march": 3,
    "apr": 4, "april": 4,
    "may": 5, "may": 5,
    "jun": 6, "june": 6,
    "jul": 7, "july": 7,
    "aug": 8, "august": 8,
    "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10,
    "nov": 11, "november": 11,
    "dec": 12, "december": 12,
}


# Slash-separated dates are interpreted as month/day/year.
# Unsupported or invalid dates become NaT and stop the export during validation.
def parse_date(value):
    """Parse platform dates without depending on the JupyterLite locale."""
    text = unicodedata.normalize("NFKC", str(value))
    text = text.replace("\u00a0", " ").strip()

    if not text:
        return pd.NaT

    # M/D/YYYY, with or without a time after it.
    match = re.search(
        r"(?<!\d)(\d{1,2})[/-](\d{1,2})[/-](\d{4})(?!\d)",
        text,
    )
    if match:
        month, day, year = map(int, match.groups())
        try:
            return pd.Timestamp(year=year, month=month, day=day)
        except ValueError:
            return pd.NaT

    # YYYY-MM-DD.
    match = re.search(
        r"(?<!\d)(\d{4})-(\d{1,2})-(\d{1,2})(?!\d)",
        text,
    )
    if match:
        year, month, day = map(int, match.groups())
        try:
            return pd.Timestamp(year=year, month=month, day=day)
        except ValueError:
            return pd.NaT

    # Sep 1, 2026 or September 1, 2026.
    match = re.search(
        r"([A-Za-z]+)\s+(\d{1,2})(?:st|nd|rd|th)?[,]?\s+(\d{4})",
        text,
        flags=re.IGNORECASE,
    )
    if match:
        month_text, day, year = match.groups()
        month = MONTH_NUMBERS.get(month_text.lower())
        if month:
            try:
                return pd.Timestamp(
                    year=int(year),
                    month=month,
                    day=int(day),
                )
            except ValueError:
                return pd.NaT

    return pd.NaT


def parse_date_column(series):
    return pd.Series(
        [parse_date(value) for value in series],
        dtype="datetime64[ns]",
    )


def standardize_post_type(value, replacements):
    """Replace known post types and preserve any unlisted raw value."""
    raw_value = str(value).strip()
    return replacements.get(raw_value.lower(), raw_value)




## 3. Show extraction and Source lookup


In [ ]:
# ==================================================
# 3. Show extraction and Source lookup
# ==================================================

def normalize_lookup_text(value):
    text = unicodedata.normalize("NFKC", str(value))
    text = text.replace("\u00a0", " ")
    return re.sub(r"\s+", " ", text).strip()


# All sources.csv is read only.
sources_lookup = read_csv_file(SOURCES_FILE)
sources_lookup["Series Title"] = (
    sources_lookup["Series Title"].map(normalize_lookup_text)
)
sources_lookup["Sources"] = sources_lookup["Sources"].astype(str).str.strip()

# Duplicate normalized show names use the first reference-table entry.
# Matching is exact after Unicode and whitespace normalization; it is not fuzzy
# matching or machine learning. Maintain the reference table and review exceptions.
source_dictionary = (
    sources_lookup.loc[sources_lookup["Series Title"].ne("")]
    .drop_duplicates("Series Title", keep="first")
    .set_index("Series Title")["Sources"]
    .to_dict()
)


# Labels depend on paired delimiters in the title. The final complete pair
# is used. Missing pairs produce a blank Show; unmatched lookups produce a blank
# Source. These blanks are retained for human review rather than guessed.
def extract_show(title):
    """
    Equivalent to TEXTBEFORE(TEXTAFTER(Title, "«", -1), "»").
    Returns blank when no complete «...» pair exists.
    """
    matches = re.findall(r"«([^»]+)»", str(title))
    return normalize_lookup_text(matches[-1]) if matches else ""


def add_show_and_source(dataframe):
    dataframe["Show"] = dataframe["Title"].map(extract_show)
    dataframe["Source"] = dataframe["Show"].map(source_dictionary).fillna("")
    return dataframe


# These platform-specific placeholders intentionally remain zero to preserve
# the shared export schema. They do not assert that the metric was measured.
def blank_platform_specific_columns(dataframe):
    dataframe["3-second video views from Boosted posts"] = 0
    dataframe["3-second video views from Organic posts"] = 0
    return dataframe




## 4. TikTok


In [ ]:
# ==================================================
# 4. TikTok
# ==================================================

tiktok_raw = read_csv_file(FILES["TikTok"]).reset_index(drop=True)

tiktok_like = number_column(tiktok_raw, "Total reactions")
tiktok_comments = number_column(tiktok_raw, "Total comments")
tiktok_shares = number_column(tiktok_raw, "Total shares")
tiktok_saves = pd.Series(0, index=tiktok_raw.index, dtype="float64")

tiktok = pd.DataFrame({
    "Date": parse_date_column(tiktok_raw["Date"]),
    "platform": "TikTok",
    "Post type": tiktok_raw["Media type"].map(
        lambda value: standardize_post_type(value, {"video": "Video"})
    ),
    "Title": tiktok_raw["Content"],
    "URL": tiktok_raw["View on platform"],
    "IMP": number_column(tiktok_raw, "Total impressions"),
    "Reach": 0,
    "Video view count": number_column(tiktok_raw, "Video view count"),
    "Total interactions": (
        tiktok_like + tiktok_comments + tiktok_shares + tiktok_saves
    ),
    "Like": tiktok_like,
    "comments": tiktok_comments,
    "share": tiktok_shares,
    "Save": tiktok_saves,
    "Video length (sec)": number_column(tiktok_raw, "Video length (sec)"),
})

tiktok = blank_platform_specific_columns(tiktok)
tiktok = add_show_and_source(tiktok)




## 5. X


In [ ]:
# ==================================================
# 5. X
# ==================================================

x_raw = read_csv_file(FILES["X"]).reset_index(drop=True)

x_like = number_column(x_raw, "Total reactions")
x_comments = number_column(x_raw, "Total comments")
x_shares = number_column(x_raw, "Total shares")
x_saves = pd.Series(0, index=x_raw.index, dtype="float64")

x_data = pd.DataFrame({
    "Date": parse_date_column(x_raw["Date"]),
    "platform": "X",
    "Post type": x_raw["Media type"].map(
        lambda value: standardize_post_type(
            value,
            {"photo": "Image", "video": "Video", "link": "Link"},
        )
    ),
    "Title": x_raw["Content"],
    "URL": x_raw["View on platform"],
    "IMP": number_column(x_raw, "Total impressions"),
    "Reach": 0,
    "Video view count": number_column(x_raw, "Video view count"),
    "Total interactions": x_like + x_comments + x_shares + x_saves,
    "Like": x_like,
    "comments": x_comments,
    "share": x_shares,
    "Save": x_saves,
    "Video length (sec)": 0,
})

x_data = blank_platform_specific_columns(x_data)
x_data = add_show_and_source(x_data)




## 6. Facebook


In [ ]:
# ==================================================
# 6. Facebook
# ==================================================

facebook_raw = read_csv_file(FILES["Facebook"]).reset_index(drop=True)

facebook_like = number_column(facebook_raw, "Reactions")
facebook_comments = number_column(facebook_raw, "Comments")
facebook_shares = number_column(facebook_raw, "Shares")
facebook_saves = pd.Series(0, index=facebook_raw.index, dtype="float64")

facebook = pd.DataFrame({
    "Date": parse_date_column(facebook_raw["Publish time"]),
    "platform": "Facebook",
    "Post type": facebook_raw["Post type"].map(
        lambda value: standardize_post_type(
            value,
            {"photos": "Image", "videos": "Video"},
        )
    ),
    "Title": facebook_raw["Title"],
    "URL": facebook_raw["Permalink"],
    # The supplied Facebook sample has no Total impressions column.
    # This uses it automatically if a future export contains it; otherwise 0.
    "IMP": number_column(facebook_raw, "Total impressions"),
    "Reach": number_column(facebook_raw, "Reach"),
    "Video view count": number_column(facebook_raw, "3-second video views"),
    "Total interactions": (
        facebook_like + facebook_comments + facebook_shares + facebook_saves
    ),
    "Like": facebook_like,
    "comments": facebook_comments,
    "share": facebook_shares,
    "Save": facebook_saves,
    "Video length (sec)": number_column(facebook_raw, "Duration (sec)"),
    "3-second video views from Boosted posts": number_column(
        facebook_raw,
        "3-second video views from Boosted posts",
    ),
    "3-second video views from Organic posts": number_column(
        facebook_raw,
        "3-second video views from Organic posts",
    ),
})

facebook = add_show_and_source(facebook)




## 7. Instagram


In [ ]:
# ==================================================
# 7. Instagram
# ==================================================

instagram_raw = read_csv_file(FILES["Instagram"]).reset_index(drop=True)

instagram_like = number_column(instagram_raw, "Likes")
instagram_comments = number_column(instagram_raw, "Comments")
instagram_shares = number_column(instagram_raw, "Shares")
instagram_saves = number_column(instagram_raw, "Saves")

# Preserve the reporting rule: only IG Reel posts contribute video views.
# Other Instagram post types receive zero even if their export contains Views.
instagram_views = number_column(instagram_raw, "Views").where(
    instagram_raw["Post type"].str.strip().str.lower().eq("ig reel"),
    0,
)

instagram = pd.DataFrame({
    "Date": parse_date_column(instagram_raw["Publish time"]),
    "platform": "Instagram",
    "Post type": instagram_raw["Post type"].map(
        lambda value: standardize_post_type(
            value,
            {
                "ig carousel": "Carousel",
                "ig reel": "Reel",
                "ig image": "Image",
            },
        )
    ),
    "Title": instagram_raw["Description"],
    "URL": instagram_raw["Permalink"],
    "IMP": 0,
    "Reach": number_column(instagram_raw, "Reach"),
    "Video view count": instagram_views,
    "Total interactions": (
        instagram_like
        + instagram_comments
        + instagram_shares
        + instagram_saves
    ),
    "Like": instagram_like,
    "comments": instagram_comments,
    "share": instagram_shares,
    "Save": instagram_saves,
    "Video length (sec)": number_column(instagram_raw, "Duration (sec)"),
})

instagram = blank_platform_specific_columns(instagram)
instagram = add_show_and_source(instagram)




## 8. YouTube


In [ ]:
# ==================================================
# 8. YouTube
# ==================================================

youtube_raw = read_csv_file(FILES["YouTube"])

# Safe even when you have already removed the Total row manually.
youtube_raw = youtube_raw.loc[
    youtube_raw["Content"].str.strip().str.lower().ne("total")
].reset_index(drop=True)

# YouTube Total interactions includes dislikes as well as likes, comments,
# and shares. Other platforms follow their existing interaction formulas.
youtube_likes = number_column(youtube_raw, "Likes")
youtube_dislikes = number_column(youtube_raw, "Dislikes")
youtube_comments = number_column(youtube_raw, "Comments added")
youtube_shares = number_column(youtube_raw, "Shares")
youtube_saves = pd.Series(0, index=youtube_raw.index, dtype="float64")

youtube = pd.DataFrame({
    "Date": parse_date_column(youtube_raw["Video publish time"]),
    "platform": "YouTube",
    "Post type": "Video",
    "Title": youtube_raw["Video title"],
    "URL": (
        "https://www.youtube.com/watch?v="
        + youtube_raw["Content"].str.strip()
    ),
    "IMP": number_column(youtube_raw, "Thumbnail impressions"),
    "Reach": 0,
    "Video view count": number_column(youtube_raw, "Views"),
    "Total interactions": (
        youtube_likes
        + youtube_dislikes
        + youtube_comments
        + youtube_shares
    ),
    "Like": youtube_likes,
    "comments": youtube_comments,
    "share": youtube_shares,
    "Save": youtube_saves,
    "Video length (sec)": number_column(youtube_raw, "Duration"),
})

youtube = blank_platform_specific_columns(youtube)
youtube = add_show_and_source(youtube)




## 9. Combine and validate

The common schema is an internal reporting convenience. Review the platform-specific mappings and agreed formulas before treating any totals as comparable measures.


In [ ]:
# ==================================================
# 9. Combine and validate
# ==================================================

# Standardized column names do not make platform metric definitions identical.
# For example: Facebook 3-second views, YouTube views, and YouTube thumbnail
# impressions should be interpreted using their respective source definitions.
weekly_report = pd.concat(
    [tiktok, x_data, facebook, instagram, youtube],
    ignore_index=True,
)

weekly_report = weekly_report[OUTPUT_COLUMNS]

# Guarantee that every requested metric is numeric and never blank.
for column in NUMERIC_COLUMNS:
    weekly_report[column] = pd.to_numeric(
        weekly_report[column],
        errors="coerce",
    ).fillna(0)

invalid_dates = weekly_report["Date"].isna()
if invalid_dates.any():
    invalid_rows = weekly_report.loc[
        invalid_dates,
        ["platform", "Title"],
    ]
    raise ValueError(
        "The dates could not be identified for these rows:\n"
        + invalid_rows.to_string(index=True)
    )

weekly_report = weekly_report.sort_values(
    ["Date", "platform", "Title"],
    ascending=[True, True, True],
).reset_index(drop=True)




## 10. Create dated output folder


In [ ]:
# ==================================================
# 10. Create dated output folder
# ==================================================

# Folder naming uses the observed publication-date range, not an explicit
# reporting-week filter. The existing name omits the year and repeated runs
# overwrite the same CSV path; retain this behavior for output compatibility.
minimum_date = weekly_report["Date"].min()
maximum_date = weekly_report["Date"].max()

MONTH_LABELS = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",
    5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
    9: "Sept", 10: "Oct", 11: "Nov", 12: "Dec",
}

if minimum_date.month == maximum_date.month:
    date_label = (
        f"{minimum_date.day:02d}-{maximum_date.day:02d}"
        f"{MONTH_LABELS[maximum_date.month]}"
    )
else:
    date_label = (
        f"{minimum_date.day:02d}{MONTH_LABELS[minimum_date.month]}-"
        f"{maximum_date.day:02d}{MONTH_LABELS[maximum_date.month]}"
    )

weekly_folder_name = f"{date_label} Weekly"
output_folder = OUTPUT_ROOT / weekly_folder_name
output_folder.mkdir(parents=True, exist_ok=True)
output_file = output_folder / f"{weekly_folder_name}.csv"




## 11. Format, export, and download


In [ ]:
# ==================================================
# 11. Format, export, and download
# ==================================================

weekly_report["Date"] = weekly_report["Date"].map(
    lambda value: f"{value.month}/{value.day}/{value.year}"
)

weekly_report.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig",
)

# Compatibility note: the existing printed Show/Source count checks Show only.
# A nonblank Show can still have a blank Source if the lookup has no match.
# Review both columns manually; the printed count is not full label validation.
blank_show_count = weekly_report["Show"].eq("").sum()

print("Weekly report created successfully.")
print(f"Output file: {output_file}")
print(f"Total rows: {len(weekly_report)}")
print(f"Rows with a blank Show/Source: {blank_show_count}")

display(weekly_report.head())
display(FileLink(str(output_file)))

## Selected metric totals

These totals preserve the original checks. Interpret platform-specific metric definitions before comparing or aggregating them.

Cross-platform sums reproduce the internal report convention; they are not a universal measurement standard.


In [ ]:
weekly_report[["IMP", "Video view count", "Total interactions"]].sum()

## Totals by platform

Displays formatted totals for review; does not alter the exported dataset.


In [ ]:
weekly_report.groupby("platform")[["IMP", "Video view count", "Total interactions"]].sum().map(lambda x: f"{x:,.0f}")

## Null-value check

Empty strings are not counted as nulls. Zero-filled numerical fields also pass this check even when the source value was unavailable.


In [ ]:
weekly_report.isnull().sum()

## Duplicate URL count

Counts all rows belonging to duplicate URL groups, including repeated blanks. This is a review check, not automatic deduplication.


In [ ]:
weekly_report.duplicated(subset=['URL'], keep=False).sum()

## Duplicate URL review

Inspect the records before deciding whether they are duplicates. This cell does not remove any rows.


In [ ]:
weekly_report[weekly_report.duplicated(subset=['URL'], keep=False)]

## Total record count

Confirms the number of rows in the consolidated dataset.


In [ ]:
weekly_report['platform'].size